In [1]:
from pathlib import Path
import duckdb

DATA = Path("data")
con = duckdb.connect()
con.execute("SET memory_limit = '8GB'")

# Register Parquet files as views (no data copied into RAM)
for name in ["articles", "customers", "transactions_train"]:
    con.execute(f"CREATE VIEW {name} AS SELECT * FROM '{(DATA / f'{name}.parquet').as_posix()}'")

print(con.execute("SHOW TABLES").df())

                 name
0            articles
1           customers
2  transactions_train


In [2]:
import pandas as pd
pd.set_option("display.max_rows", 50)

for name in ["articles", "customers", "transactions_train"]:
    df = con.execute(f"SUMMARIZE {name}").df()
    df = df[["column_name", "column_type", "null_percentage", "approx_unique"]]
    missing = df[df["null_percentage"].astype(float) > 0]
    print(f"\n=== {name} ===  columns with missing values: {len(missing)}")
    print(missing.to_string(index=False) if len(missing) else "none")


=== articles ===  columns with missing values: 1
column_name column_type  null_percentage  approx_unique
detail_desc     VARCHAR             0.39          50802

=== customers ===  columns with missing values: 5
           column_name column_type  null_percentage  approx_unique
                    FN      DOUBLE            65.24              1
                Active      DOUBLE            66.15              1
    club_member_status     VARCHAR             0.44              3
fashion_news_frequency     VARCHAR             1.17              4
                   age      BIGINT             1.16             78


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== transactions_train ===  columns with missing values: 0
none


In [3]:
for col in ["FN", "Active", "club_member_status", "fashion_news_frequency"]:
    print(f"\n=== {col} ===")
    print(con.execute(f"""
        SELECT {col} AS value, COUNT(*) AS customers,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
        FROM customers
        GROUP BY {col}
        ORDER BY customers DESC
    """).df().to_string(index=False))


=== FN ===
 value  customers   pct
   NaN     895050 65.24
   1.0     476930 34.76

=== Active ===
 value  customers   pct
   NaN     907576 66.15
   1.0     464404 33.85

=== club_member_status ===
     value  customers   pct
    ACTIVE    1272491 92.75
PRE-CREATE      92960  6.78
       NaN       6062  0.44
 LEFT CLUB        467  0.03

=== fashion_news_frequency ===
    value  customers   pct
     NONE     877711 63.97
Regularly     477416 34.80
      NaN      16009  1.17
  Monthly        842  0.06
     None          2  0.00


In [4]:
print(con.execute("""
    WITH counts AS (
        SELECT t_dat, customer_id, article_id, price, sales_channel_id,
               COUNT(*) AS n
        FROM transactions_train
        GROUP BY ALL
    )
    SELECT
        SUM(n)                        AS total_rows,
        COUNT(*)                      AS unique_rows,
        SUM(n) - COUNT(*)             AS duplicate_rows,
        ROUND(100.0 * (SUM(n) - COUNT(*)) / SUM(n), 2) AS dup_pct,
        MAX(n)                        AS max_copies
    FROM counts
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 total_rows  unique_rows  duplicate_rows  dup_pct  max_copies
 31788324.0     28813419       2974905.0     9.36         569


In [5]:
print(con.execute("""
    WITH counts AS (
        SELECT t_dat, customer_id, article_id, price, COUNT(*) AS n
        FROM transactions_train
        GROUP BY ALL
    )
    SELECT c.t_dat, LEFT(c.customer_id, 10) AS customer, c.article_id,
           a.prod_name, c.n AS copies
    FROM counts c
    LEFT JOIN articles a USING (article_id)
    ORDER BY c.n DESC
    LIMIT 10
""").df().to_string(index=False))

print(con.execute("""
    WITH counts AS (
        SELECT customer_id, article_id, t_dat, COUNT(*) AS n
        FROM transactions_train GROUP BY ALL
    )
    SELECT n AS copies, COUNT(*) AS groups
    FROM counts
    GROUP BY n ORDER BY n LIMIT 6
""").df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     t_dat   customer article_id                 prod_name  copies
2018-10-14 d00063b94d 0678342001                  Lima SS.     569
2019-02-16 94665b46e1 0629420001                  claudine     199
2019-02-11 8f5f1e993e 0189634001         Long Leg Leggings     120
2019-10-16 0152964ef1 0756322001        Lindsay N-slip (W)     120
2019-09-28 5cba04ed9a 0688558002               Brittany LS     117
2020-02-07 c163adbcac 0689365047        BROADWAY SHIRT S.1     114
2019-02-23 61da44a275 0507909001 Rebecca or Delphine shirt     105
2018-09-24 a81e0b7657 0618480001   Dean Super Skinny Twill     100
2019-07-16 18d346ffea 0719348003                Bellora(1)     100
2019-01-17 ef38ec0f0c 0570002001       ROY SLIM RN T-SHIRT      89


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 copies   groups
      1 25857551
      2  2418787
      3   205350
      4    63677
      5    10207
      6    10013


In [6]:
print(con.execute("""
    SELECT
        MIN(price) AS min_price,
        ROUND(QUANTILE_CONT(price, 0.5), 4) AS median_price,
        MAX(price) AS max_price,
        SUM(CASE WHEN price <= 0 THEN 1 ELSE 0 END) AS zero_or_negative
    FROM transactions_train
""").df().to_string(index=False))

print(con.execute("""
    SELECT
        MIN(age) AS min_age,
        QUANTILE_CONT(age, 0.5) AS median_age,
        MAX(age) AS max_age,
        SUM(CASE WHEN age < 16 THEN 1 ELSE 0 END) AS under_16,
        SUM(CASE WHEN age > 90 THEN 1 ELSE 0 END) AS over_90
    FROM customers
""").df().to_string(index=False))

 min_price  median_price  max_price  zero_or_negative
  0.000017        0.0254   0.591525               0.0
 min_age  median_age  max_age  under_16  over_90
      16        32.0       99       0.0     77.0


In [7]:
print(con.execute("""
    SELECT
        (SELECT COUNT(*) FROM transactions_train t
         LEFT JOIN customers c USING (customer_id)
         WHERE c.customer_id IS NULL) AS tx_without_customer,
        (SELECT COUNT(*) FROM transactions_train t
         LEFT JOIN articles a USING (article_id)
         WHERE a.article_id IS NULL) AS tx_without_article,
        (SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers) AS dup_customer_ids,
        (SELECT COUNT(*) - COUNT(DISTINCT article_id) FROM articles) AS dup_article_ids
""").df().to_string(index=False))

 tx_without_customer  tx_without_article  dup_customer_ids  dup_article_ids
                   0                   0                 0                0


## Data quality: what I found

The transactions table (31,788,324 rows) is clean. No missing values, and every row matches a real customer and a real product.

About 9.4% of transaction rows are exact copies. At first I thought they were errors. Then I realised there's no quantity column, so buying two of the same thing just shows up as two rows. Around 90% of purchases are single items anyway. A few cases are odd though. One customer bought the same item 569 times in one day. That's probably a business, not a normal shopper. I kept all rows, but when I count popularity I'll count unique buyers, not rows. Otherwise one bulk buyer could make an item look popular.

In the customer table, `FN` and `Active` only contain 1 or nothing. I'm treating "nothing" as 0. H&M didn't explain these columns, so that's my guess, not a fact. `fashion_news_frequency` has "NONE" and "None" as separate values. I'll merge them. Ages go from 16 to 99, median 32, and about 1.2% are missing.

Around 0.4% of products have no description. I'll use the product name instead when I build text embeddings.

Prices are scaled by H&M, so they're not real euros. None are zero or negative.

One thing to remember for later: 9,699 customers never bought anything. They'll need their own approach (cold start).